
# Notebook 14: Scikit-learn Preprocessing Pipeline

## Objective
Learn how to build automated preprocessing pipelines using
Scikit-learn.

## Learning Outcomes
- Understand why preprocessing pipelines are required.
- Learn Pipeline and ColumnTransformer.
- Build numerical and categorical pipelines.
- Apply imputation, scaling, and encoding.
- Combine multiple preprocessing transformations.
- Understand fit(), transform(), and fit_transform().
- Build a complete ML-ready preprocessing workflow.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
np.random.seed(42)

data = {
    "Age": [22, 25, 30, 35, np.nan, 45, 50, 28, 32, 40],
    "Salary": [25000, 30000, 45000, np.nan, 60000, 75000, 90000, 35000, 50000, 65000],
    "Experience": [1, 2, 5, 8, 10, np.nan, 15, 3, 6, 12],
    "Department": [
        "IT", "HR", "Finance", "IT", "HR",
        "Finance", "IT", "HR", np.nan, "Finance"
    ],
    "Education": [
        "Graduate", "Postgraduate", "Graduate",
        "Graduate", "Postgraduate", "PhD",
        "Graduate", "Graduate", "Postgraduate", "PhD"
    ],
    "Gender": [
        "Male", "Female", "Male", "Female", "Male",
        "Female", "Male", "Female", "Male", "Female"
    ]
}

df = pd.DataFrame(data)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head(10))

Dataset loaded successfully!
Dataset shape: (10, 6)


,Age,Salary,Experience,Department,Education,Gender
0,22.0,25000.0,1.0,IT,Graduate,Male
1,25.0,30000.0,2.0,HR,Postgraduate,Female
2,30.0,45000.0,5.0,Finance,Graduate,Male
3,35.0,NaN,8.0,IT,Graduate,Female
4,NaN,60000.0,10.0,HR,Postgraduate,Male
5,45.0,75000.0,NaN,Finance,PhD,Female
6,50.0,90000.0,15.0,IT,Graduate,Male
7,28.0,35000.0,3.0,HR,Graduate,Female
8,32.0,50000.0,6.0,NaN,Postgraduate,Male
9,40.0,65000.0,12.0,Finance,PhD,Female


In [3]:
print("Dataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing_Count"))

print("\nNumerical columns:")
print(df.select_dtypes(include="number").columns.tolist())

print("\nCategorical columns:")
print(df.select_dtypes(include=["object", "category", "str"]).columns.tolist())

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Age         9 non-null      float64
 1   Salary      9 non-null      float64
 2   Experience  9 non-null      float64
 3   Department  9 non-null      str    
 4   Education   10 non-null     str    
 5   Gender      10 non-null     str    
dtypes: float64(3), str(3)
memory usage: 779.0 bytes

Missing values:


,Missing_Count
Age,1
Salary,1
Experience,1
Department,1
Education,0
Gender,0



Numerical columns:
['Age', 'Salary', 'Experience']

Categorical columns:
['Department', 'Education', 'Gender']



## 1. Why Use Pipelines?

A preprocessing pipeline combines multiple data-processing
operations into one organized workflow.

Benefits:

- Automates preprocessing steps.
- Reduces repetitive code.
- Maintains consistent transformations.
- Helps prevent data leakage.
- Simplifies model development.
- Makes cross-validation easier.
- Improves code organization and reproducibility.

Instead of manually applying every transformation, we can
define the complete workflow once and reuse it.



## 2. Pipeline

Pipeline is a Scikit-learn tool that connects multiple
transformations in a sequential order.

Example:

Raw Data
   ↓
Missing Value Imputation
   ↓
Feature Scaling
   ↓
Processed Data

Each intermediate step must implement fit() and transform().
The final step can be a transformer or an estimator.


In [4]:
numerical_features = ["Age", "Salary", "Experience"]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline created successfully!")

numerical_pipeline

Numerical pipeline created successfully!


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

In [5]:
numerical_data = df[numerical_features]

numerical_transformed = numerical_pipeline.fit_transform(
    numerical_data
)

df_numerical_processed = pd.DataFrame(
    numerical_transformed,
    columns=numerical_features
)

print("Numerical preprocessing completed!")

display(df_numerical_processed.head())

Numerical preprocessing completed!


,Age,Salary,Experience
0,-1.423441,-1.417733,-1.361037
1,-1.064590,-1.159963,-1.126376
2,-0.466506,-0.386654,-0.422391
3,0.131579,-0.128885,0.281594
4,-0.227272,0.386654,0.750917



## 3. Numerical Pipeline Explanation

### SimpleImputer(strategy="median")
- Detects missing numerical values.
- Calculates the median from the training data.
- Replaces missing values with the calculated median.

### StandardScaler()
- Calculates the mean and standard deviation.
- Centers each feature around zero.
- Scales features to approximately unit variance.

The order is important: missing values must be handled before
applying StandardScaler.


In [6]:
categorical_features = [
    "Department",
    "Education",
    "Gender"
]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical pipeline created successfully!")

categorical_pipeline

Categorical pipeline created successfully!


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto ac

In [7]:
categorical_data = df[categorical_features]

categorical_transformed = categorical_pipeline.fit_transform(
    categorical_data
)

encoded_columns = (
    categorical_pipeline
    .named_steps["encoder"]
    .get_feature_names_out(categorical_features)
)

df_categorical_processed = pd.DataFrame(
    categorical_transformed.toarray(),
    columns=encoded_columns
)

print("Categorical preprocessing completed!")

display(df_categorical_processed.head())

Categorical preprocessing completed!


,Department_Finance,Department_HR,Department_IT,Education_Graduate,Education_PhD,Education_Postgraduate,Gender_Female,Gender_Male
0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0



## 4. Categorical Pipeline Explanation

### SimpleImputer(strategy="most_frequent")
- Identifies missing categorical values.
- Finds the most frequently occurring category.
- Replaces missing values with that category.

### OneHotEncoder(handle_unknown="ignore")
- Converts categories into binary numerical columns.
- Creates separate columns for different categories.
- Handles previously unseen categories during transformation
  without producing an error.

Example:

Department = IT, HR, Finance

After encoding:

Department_Finance
Department_HR
Department_IT

One-hot encoding avoids introducing an artificial numerical
ordering between nominal categories.



## 5. ColumnTransformer

ColumnTransformer allows us to apply different preprocessing
operations to different columns of the same dataset.

For example:

Numerical Features:
Age, Salary, Experience
        ↓
Median Imputation
        ↓
Standard Scaling

Categorical Features:
Department, Education, Gender
        ↓
Most Frequent Imputation
        ↓
One-Hot Encoding

Both processed outputs are combined into one transformed dataset.


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("ColumnTransformer created successfully!")

preprocessor

ColumnTransformer created successfully!


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

In [9]:
complete_pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

print("Complete preprocessing pipeline created!")

complete_pipeline

Complete preprocessing pipeline created!


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the o


## 6. Complete Preprocessing Workflow

Our complete pipeline follows this structure:

RAW DATA
   |
   |--- Numerical Features
   |       |
   |       |--- Median Imputation
   |       |
   |       |--- Standard Scaling
   |
   |--- Categorical Features
           |
           |--- Most Frequent Imputation
           |
           |--- One-Hot Encoding
           
              |
              ↓
       COMBINED PROCESSED DATA

The ColumnTransformer applies the appropriate transformations
to each feature group and combines the results.


In [10]:
X_train, X_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print("Training dataset shape:", X_train.shape)
print("Testing dataset shape:", X_test.shape)

print("\nTraining missing values:")
display(X_train.isnull().sum())

print("\nTesting missing values:")
display(X_test.isnull().sum())

Training dataset shape: (8, 6)
Testing dataset shape: (2, 6)

Training missing values:


Age           1
Salary        1
Experience    1
Department    0
Education     0
Gender        0
dtype: int64


Testing missing values:


Age           0
Salary        0
Experience    0
Department    1
Education     0
Gender        0
dtype: int64

In [11]:
# Fit preprocessing on training data and transform it.
X_train_processed = complete_pipeline.fit_transform(X_train)

# Transform test data using the already fitted pipeline.
X_test_processed = complete_pipeline.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

print("\nPreprocessing completed successfully!")

Training processed shape: (8, 11)
Testing processed shape: (2, 11)

Preprocessing completed successfully!



## 7. fit()

The fit() method learns the necessary parameters from the data.

Examples:
- SimpleImputer learns median or mode.
- StandardScaler learns mean and standard deviation.
- OneHotEncoder learns available categories.

## 8. transform()

The transform() method applies previously learned parameters
to new data.

It does not learn new parameters.

## 9. fit_transform()

The fit_transform() method combines fit() and transform().

It learns parameters from the supplied data and transforms
that same data.

Recommended usage:

Training Data:
fit_transform()

Testing Data:
transform()


In [12]:
demo_scaler = StandardScaler()

demo_scaler.fit(X_train[numerical_features])

print("Learned means:")
print(demo_scaler.mean_)

print("\nLearned standard deviations:")
print(demo_scaler.scale_)

Learned means:
[3.57142857e+01 5.64285714e+04 7.71428571e+00]

Learned standard deviations:
[9.17627820e+00 2.11650463e+04 4.65109160e+00]


In [13]:
transformed_demo = demo_scaler.transform(
    X_test[numerical_features]
)

df_transformed_demo = pd.DataFrame(
    transformed_demo,
    columns=numerical_features
)

print("Test data after transformation:")
display(df_transformed_demo.head())

Test data after transformation:


,Age,Salary,Experience
0,-0.404770,-0.303735,-0.368577
1,-1.167607,-1.248690,-1.228590


In [14]:
demo_scaler_2 = StandardScaler()

fit_transform_demo = demo_scaler_2.fit_transform(
    X_train[numerical_features]
)

df_fit_transform_demo = pd.DataFrame(
    fit_transform_demo,
    columns=numerical_features
)

print("Training data after fit_transform():")
display(df_fit_transform_demo.head())

Training data after fit_transform():


,Age,Salary,Experience
0,1.011926,0.877457,NaN
1,-1.494537,-1.484928,-1.443594
2,-0.840677,-1.012451,-1.013587
3,-0.622724,-0.539974,-0.583580
4,0.467043,0.404980,0.921443


In [15]:
# Get transformed feature names.
feature_names = complete_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

# Convert sparse output to a dense array when necessary.
if hasattr(X_train_processed, "toarray"):
    train_array = X_train_processed.toarray()
else:
    train_array = np.asarray(X_train_processed)

df_final_processed = pd.DataFrame(
    train_array,
    columns=feature_names,
    index=X_train.index
)

print("Final processed dataset:")
print("Shape:", df_final_processed.shape)

display(df_final_processed.head())

Final processed dataset:
Shape: (8, 11)


,numerical__Age,numerical__Salary,numerical__Experience,categorical__Department_Finance,categorical__Department_HR,categorical__Department_IT,categorical__Education_Graduate,categorical__Education_PhD,categorical__Education_Postgraduate,categorical__Gender_Female,categorical__Gender_Male
5,1.091783,0.913868,0.057448,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
0,-1.586724,-1.607146,-1.551109,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
7,-0.887983,-1.102944,-1.091521,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-0.655070,-0.598741,-0.631933,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
9,0.509499,0.409665,0.976624,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [16]:
print("Missing values after preprocessing:")
print(df_final_processed.isnull().sum().sum())

print("\nAny remaining missing values?",
      df_final_processed.isnull().values.any())

print("\nAll features numerical?",
      all(pd.api.types.is_numeric_dtype(dtype)
          for dtype in df_final_processed.dtypes))

print("\nFinal verification completed!")

Missing values after preprocessing:
0

Any remaining missing values? False

All features numerical? True

Final verification completed!


In [17]:
print("Original training dataset:")
display(X_train.head())

print("\nProcessed training dataset:")
display(df_final_processed.head())

print("\nOriginal shape:", X_train.shape)
print("Processed shape:", df_final_processed.shape)

Original training dataset:


,Age,Salary,Experience,Department,Education,Gender
5,45.0,75000.0,NaN,Finance,PhD,Female
0,22.0,25000.0,1.0,IT,Graduate,Male
7,28.0,35000.0,3.0,HR,Graduate,Female
2,30.0,45000.0,5.0,Finance,Graduate,Male
9,40.0,65000.0,12.0,Finance,PhD,Female



Processed training dataset:


,numerical__Age,numerical__Salary,numerical__Experience,categorical__Department_Finance,categorical__Department_HR,categorical__Department_IT,categorical__Education_Graduate,categorical__Education_PhD,categorical__Education_Postgraduate,categorical__Gender_Female,categorical__Gender_Male
5,1.091783,0.913868,0.057448,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
0,-1.586724,-1.607146,-1.551109,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
7,-0.887983,-1.102944,-1.091521,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-0.655070,-0.598741,-0.631933,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
9,0.509499,0.409665,0.976624,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0



Original shape: (8, 6)
Processed shape: (8, 11)



## 10. Advantages of Preprocessing Pipelines

1. Automation:
   Multiple preprocessing steps execute automatically.

2. Consistency:
   The same transformations are applied to training and testing data.

3. Leakage Prevention:
   Learned preprocessing parameters are fitted on training data.

4. Reusability:
   Pipelines can be reused for different datasets and models.

5. Maintainability:
   The preprocessing workflow is organized and easier to manage.

## Limitations

- Complex pipelines can be difficult to debug.
- Incorrect column selection can cause errors.
- Choosing inappropriate imputation or encoding methods
  can reduce model quality.
- Pipelines do not automatically prevent every type of leakage.
  Feature creation and data splitting must also be correct.



## Conclusion

In this notebook, I learned how to create preprocessing pipelines
using Scikit-learn.

I implemented numerical preprocessing using median imputation
and StandardScaler, and categorical preprocessing using
most-frequent imputation and OneHotEncoder.

I used ColumnTransformer to combine different transformations
and Pipeline to organize the complete preprocessing workflow.

I also practiced fit(), transform(), and fit_transform() and
understood why preprocessing should be fitted only on training data.

Finally, I generated a clean, consistent, numerical dataset
that is ready for machine learning model development.
